# Level 3 AI Toy Demo

This notebook is a runnable demo wrapper around the Python implementation.

The production-style files are still:

- `openclaw_adapter.py`
- `pi_service.py`
- `mission_control.py`
- `vision.py`

Use this notebook for screenshots, walkthroughs, and quick end-to-end validation.

## 1. In-process smoke test

This cell validates mission parsing and vision output without starting the HTTP service.

In [ ]:
import json

from mission_control import parse_instruction
from vision import run_vision_task

instruction = "Check whether there is a person in the image"
plan = parse_instruction(instruction)
vision = run_vision_task(plan.target, plan.source_image)

result = {
    "success": vision.found_target,
    "mission_plan": plan.to_dict(),
    "vision_result": vision.to_dict(),
    "outcome": f"Affirmative: completed {plan.intent} for {plan.target}.",
}

print(json.dumps(result, indent=2))

## 2. HTTP end-to-end demo

This cell starts the Raspberry Pi mock service, sends a request through the Open Claw adapter, and then shuts the service down.

In [ ]:
import subprocess
import sys
import time

import requests

from openclaw_adapter import send_instruction

port = 8010
service_url = f"http://127.0.0.1:{port}"

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "pi_service:app", "--host", "127.0.0.1", "--port", str(port)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

try:
    for _ in range(30):
        try:
            health = requests.get(f"{service_url}/health", timeout=1)
            if health.status_code == 200:
                break
        except requests.RequestException:
            time.sleep(0.5)
    else:
        raise RuntimeError("Pi service did not become healthy.")

    response = send_instruction(
        "Check whether there is a person in the image",
        service_url=service_url,
    )
    print(json.dumps(response, indent=2))
finally:
    server.terminate()
    try:
        server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server.kill()

## 3. Optional next steps

- Add a real image path with `source_image="sample_images/your_image.jpg"`.
- Install `requirements-api.txt` and configure `.env` for API mission control.
- Install `requirements-vision.txt` to run YOLO on local images.
- Deploy `pi_service.py` to a real Raspberry Pi later without changing the adapter contract.